In [1]:
import pandas as pd
import numpy as np

In [ ]:
ripple_4k7_100n_12 = 140e-3
ripple_10k_100n_12 = 65.65e-3
ripple_4k7_100n_37 = 45.9e-3
ripple_10k_100n_37 = 21.58e-3

In [5]:
tau_4k7_100n = (4.7e3 * 100e-9)
tau_10k_100n = (10e3 * 100e-9)
print(tau_4k7_100n, tau_10k_100n)

0.00047 0.001


In [9]:
data = {
    "Wrap": [4000, 8000, 12000, 16000],
}
df = pd.DataFrame(data)
df["Freq[Hz]"] = 150e6 / df["Wrap"]
df["Ripple_10k_100n"] = 3.3 * (1 - np.exp(- 1 / (df["Freq[Hz]"] * tau_10k_100n)))
df

,Wrap,Freq[Hz],Ripple_10k_100n
0,4000,37500.0,0.086837
1,8000,18750.0,0.171389
2,12000,12500.0,0.253716
3,16000,9375.0,0.333877


## FPB Sallen Key
- Orden 2
- Ganancia N
- Freq corte: 2k
- Freq atenuacion: 8k
- $A = \frac{fc}{fat}$

In [ ]:
fc = 720
rn = 4.7e3
H0 = 1
a = np.sqrt(2)
b = 1

def calculate_filter_components(H0, a = np.sqrt(2), b = 1):
    r1n = r2n = r3n = 1
    r4n = H0 - 1
    c1n = (a + np.sqrt(a**2 + 8 * b * (H0 - 1))) / (4 * b)
    c2n = 4 / (a + np.sqrt(a**2 + 8 * b * (H0 - 1)))
    return r1n, r2n, r3n, r4n, c1n, c2n

normal_values = calculate_filter_components(H0, a, b)
df = pd.DataFrame([normal_values], columns=["R1n", "R2n", "R3n", "R4n", "C1n", "C2n"])
df

# c1 = normal_values[4] / (2 * np.pi * fc * rn)
# c2 = normal_values[5] / (2 * np.pi * fc * rn)


# print(f"R1: {r1n * rn / 1e3:.2f} kΩ, R4: {r4n * rn / 1e3:.2f} kΩ")
# print(f"C1: {c1 / 1e-9:.2f} nF, C2: {c2 / 1e-9:.2f} nF")

,R1n,R2n,R3n,R4n,C1n,C2n
0,1,1,1,0,0.707107,1.414214


In [3]:
## Para ganacia 1
a = np.sqrt(2)
b = 1

c1n = a / (2*b)
c2n = 2 / a
r_values = [4.7e3, 5.1e3, 6.8e3, 8.2e3, 10e3, 22e3, 33e3]
test_values = [
    (r, c1n, c2n) for r in r_values
]
df = pd.DataFrame(test_values, columns=["R", "C1", "C2"])
for freq in range(900, 2000, 100):
    df[f"C1[nF]_{freq}"] = (df["C1"] * 1e9 / (2 * np.pi * freq * df["R"])).round(2)
    df[f"C2[nF]_{freq}"] = (df["C2"] * 1e9 / (2 * np.pi * freq * df["R"])).round(2)
df

,R,C1,C2,C1[nF]_900,C2[nF]_900,C1[nF]_1000,C2[nF]_1000,C1[nF]_1100,C2[nF]_1100,C1[nF]_1200,...,C1[nF]_1500,C2[nF]_1500,C1[nF]_1600,C2[nF]_1600,C1[nF]_1700,C2[nF]_1700,C1[nF]_1800,C2[nF]_1800,C1[nF]_1900,C2[nF]_1900
0,4700.0,0.707107,1.414214,26.61,53.21,23.94,47.89,21.77,43.54,19.95,...,15.96,31.93,14.97,29.93,14.09,28.17,13.30,26.61,12.60,25.20
1,5100.0,0.707107,1.414214,24.52,49.04,22.07,44.13,20.06,40.12,18.39,...,14.71,29.42,13.79,27.58,12.98,25.96,12.26,24.52,11.61,23.23
2,6800.0,0.707107,1.414214,18.39,36.78,16.55,33.10,15.05,30.09,13.79,...,11.03,22.07,10.34,20.69,9.74,19.47,9.19,18.39,8.71,17.42
3,8200.0,0.707107,1.414214,15.25,30.50,13.72,27.45,12.48,24.95,11.44,...,9.15,18.30,8.58,17.16,8.07,16.15,7.62,15.25,7.22,14.45
4,10000.0,0.707107,1.414214,12.50,25.01,11.25,22.51,10.23,20.46,9.38,...,7.50,15.01,7.03,14.07,6.62,13.24,6.25,12.50,5.92,11.85
5,22000.0,0.707107,1.414214,5.68,11.37,5.12,10.23,4.65,9.30,4.26,...,3.41,6.82,3.20,6.39,3.01,6.02,2.84,5.68,2.69,5.38
6,33000.0,0.707107,1.414214,3.79,7.58,3.41,6.82,3.10,6.20,2.84,...,2.27,4.55,2.13,4.26,2.01,4.01,1.89,3.79,1.79,3.59


In [28]:
Vpwm_div = 10 / 32
df_pwm_resolution = pd.DataFrame([res for res in range(2000, 12100, 2000)], columns=["Resolution"])
df_pwm_resolution["Freq[kHz]"] = (150e6 / df_pwm_resolution["Resolution"]).round(2)
df_pwm_resolution["Sensibilidad[uV/bit]"] = (3.3 * Vpwm_div * 1e6 / df_pwm_resolution["Resolution"]).round(2)
df_pwm_resolution

,Resolution,Freq[kHz],Sensibilidad[uV/bit]
0,2000,75000.0,515.62
1,4000,37500.0,257.81
2,6000,25000.0,171.88
3,8000,18750.0,128.91
4,10000,15000.0,103.12
5,12000,12500.0,85.94
